### DP2 Approach 1 vs Approach 2 photometry validation

This notebook validates the final one-patch DP2 ELAIS-S1 products produced with two VIRCAM reference-catalogue strategies. The reference-catalogue construction and the direct AP1/AP2 reference-flux comparison are documented separately in `dmu2/dmu2_DP2/Monster_video_elais_ap2.ipynb`; they are not repeated here.

- **Approach 1 (AP1):** VIDEO DR5 supplies the VIRCAM reference photometry in Z, Y, J, H and Ks.
- **Approach 2 (AP2):** the calibrated Monster-to-VISTA transformations supply Z and Y, while J, H and Ks retain the same VIDEO DR5 cross-match used by AP1.
- **Test region:** tract 2877, patch 77.
- **Final catalogue product:** `object_unforced_measurement`.

The final catalogue comparison uses mutual nearest-neighbour positional matches within 0.5 arcsec because source IDs are not stable between independent pipeline runs. PSF photometry is compared in Y and Z, where the reference calibration changes, and in J as an unchanged control band. A final Z-band check compares the 6-pixel-radius LSST-VISTA circular aperture with VIDEO DR5 `Z_FLUX_APER_2`, matching the aperture choice used in the DP1 photometry diagnostics.

The VIDEO comparison is treated as a relative sanity check rather than an absolute zero-point test because the LSST-VISTA and VIDEO measurements differ in image processing, background treatment and aperture corrections.

### 1. Imports and input datasets

The two Butler repositories and the final Step 3c collections are defined explicitly. Repository paths are constructed relative to the `lsst-ir-fusion` project root so that the notebook does not depend on a user-specific absolute path.

In [1]:
from pathlib import Path

import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits

from lsst.daf.butler import Butler


def find_project_root(start=Path.cwd()):
    """Find the lsst-ir-fusion repository root from the current directory."""
    for path in (start, *start.parents):
        if (path / "dmu4").is_dir() and (path / "dmu5").is_dir():
            return path
    raise RuntimeError("Could not locate the lsst-ir-fusion project root.")


ROOT = find_project_root()

BASE = ROOT / "dmu4/dmu4_DP2/dmu4_DP2_Example"
REPO_AP1 = BASE / "data_ap1"
REPO_AP2 = BASE / "data"

VIDEO_PATH = (
    ROOT
    / "dmu5/dmu5_DP2/diagnostics/data/video-elais-dr5.fits"
)

TRACT = 2877
PATCH = 77

RUN_AP1 = "fusionStep3cDP2Full/20261015T105326Z"
RUN_AP2 = "videoStep3c/20261018T173729Z"

SKYMAP_AP1 = "lsst_cells_v2_dp2"
SKYMAP_AP2 = "lsst_cells_v2"

MATCH_RADIUS = 0.5 * u.arcsec
SNR_MIN = 20.0

for path in (REPO_AP1, REPO_AP2, VIDEO_PATH):
    if not path.exists():
        raise FileNotFoundError(path)

butler_ap1 = Butler(str(REPO_AP1))
butler_ap2 = Butler(str(REPO_AP2))

print("Project root :", ROOT)
print("AP1 run      :", RUN_AP1)
print("AP2 run      :", RUN_AP2)
print("VIDEO        :", VIDEO_PATH)

Project root : /local/scratch/es3e23/projects/lsst-ir-fusion
AP1 run      : fusionStep3cDP2Full/20261015T105326Z
AP2 run      : videoStep3c/20261018T173729Z
VIDEO        : /local/scratch/es3e23/projects/lsst-ir-fusion/dmu5/dmu5_DP2/diagnostics/data/video-elais-dr5.fits


### AP1–AP2 photometric comparison

For each matched source, the magnitude difference is defined as

$$
\Delta m = m_{\mathrm{AP2}} - m_{\mathrm{AP1}}
= -2.5 \log_{10}\left(\frac{F_{\mathrm{AP2}}}{F_{\mathrm{AP1}}}\right).
$$

A negative value means that the AP2 flux is brighter than the corresponding AP1 measurement.

The J band is used as a control because its reference photometry is unchanged between AP1 and AP2. In contrast, the Z and Y reference photometry is derived differently in Approach 2.

In [2]:
def load_unforced_catalog(butler, collection, skymap, band):
    return butler.get(
        "object_unforced_measurement",
        dataId={
            "skymap": skymap,
            "tract": TRACT,
            "patch": PATCH,
            "band": band,
        },
        collections=collection,
    )


def mutual_match(cat1, cat2):
    """Return row indices for mutual nearest neighbours within MATCH_RADIUS."""
    ra1 = np.asarray(cat1["coord_ra"], dtype=float)
    dec1 = np.asarray(cat1["coord_dec"], dtype=float)
    ra2 = np.asarray(cat2["coord_ra"], dtype=float)
    dec2 = np.asarray(cat2["coord_dec"], dtype=float)

    valid1 = np.isfinite(ra1) & np.isfinite(dec1)
    valid2 = np.isfinite(ra2) & np.isfinite(dec2)

    rows1 = np.flatnonzero(valid1)
    rows2 = np.flatnonzero(valid2)

    coord1 = SkyCoord(
        ra=ra1[valid1] * u.rad,
        dec=dec1[valid1] * u.rad,
    )
    coord2 = SkyCoord(
        ra=ra2[valid2] * u.rad,
        dec=dec2[valid2] * u.rad,
    )

    idx12, sep12, _ = coord1.match_to_catalog_sky(coord2)
    idx21, _, _ = coord2.match_to_catalog_sky(coord1)

    mutual = idx21[idx12] == np.arange(len(coord1))
    keep = mutual & (sep12 < MATCH_RADIUS)

    return (
        rows1[keep],
        rows2[idx12[keep]],
        sep12[keep],
    )


def robust_summary(values):
    values = np.asarray(values, dtype=float)
    med = np.median(values)
    nmad = 1.4826 * np.median(np.abs(values - med))
    return med, nmad, np.percentile(values, [16, 84])


def compare_psf_band(band):
    cat1 = load_unforced_catalog(
        butler_ap1, RUN_AP1, SKYMAP_AP1, band
    )
    cat2 = load_unforced_catalog(
        butler_ap2, RUN_AP2, SKYMAP_AP2, band
    )

    m1, m2, sep = mutual_match(cat1, cat2)

    f1 = np.asarray(cat1["base_PsfFlux_instFlux"], dtype=float)[m1]
    e1 = np.asarray(cat1["base_PsfFlux_instFluxErr"], dtype=float)[m1]
    q1 = np.asarray(cat1["base_PsfFlux_flag"], dtype=bool)[m1]

    f2 = np.asarray(cat2["base_PsfFlux_instFlux"], dtype=float)[m2]
    e2 = np.asarray(cat2["base_PsfFlux_instFluxErr"], dtype=float)[m2]
    q2 = np.asarray(cat2["base_PsfFlux_flag"], dtype=bool)[m2]

    valid = (
        np.isfinite(f1)
        & np.isfinite(f2)
        & np.isfinite(e1)
        & np.isfinite(e2)
        & (f1 > 0)
        & (f2 > 0)
        & (e1 > 0)
        & (e2 > 0)
        & (~q1)
        & (~q2)
    )

    sn1 = f1 / e1
    sn2 = f2 / e2
    high_sn = valid & (sn1 >= SNR_MIN) & (sn2 >= SNR_MIN)

    print(f"\n{band}-band catalogue comparison")
    print("=" * 40)
    print("AP1 objects           :", len(cat1))
    print("AP2 objects           :", len(cat2))
    print("matched               :", len(m1))
    print("median separation mas :", np.median(sep.to_value(u.mas)))

    for label, mask in [
        ("All valid matched sources", valid),
        (f"High-S/N matched sources (S/N >= {SNR_MIN:g})", high_sn),
    ]:
        ratio = f2[mask] / f1[mask]
        dmag = -2.5 * np.log10(ratio)
        med, nmad, p = robust_summary(dmag)

        print(f"\n{label}")
        print("-" * len(label))
        print("N sources             :", mask.sum())
        print("median AP2/AP1 flux   :", np.median(ratio))
        print("median Δmag AP2-AP1   :", med)
        print("NMAD Δmag             :", nmad)
        print("16-84% Δmag           :", p)
        print("median error AP2/AP1  :", np.median(e2[mask] / e1[mask]))

    return {
        "cat1": cat1,
        "cat2": cat2,
        "m1": m1,
        "m2": m2,
        "sep": sep,
        "valid": valid,
        "high_sn": high_sn,
    }


comparisons = {
    band: compare_psf_band(band)
    for band in ("Y", "Z", "J")
}


Y-band catalogue comparison
AP1 objects           : 13266
AP2 objects           : 13208
matched               : 12842
median separation mas : 6.915576323864702

All valid matched sources
-------------------------
N sources             : 6264
median AP2/AP1 flux   : 1.0089432255303632
median Δmag AP2-AP1   : -0.00966682157908317
NMAD Δmag             : 0.025884175074943117
16-84% Δmag           : [-0.04187464  0.02248278]
median error AP2/AP1  : 0.9982051220137127

High-S/N matched sources (S/N >= 20)
------------------------------------
N sources             : 2683
median AP2/AP1 flux   : 1.0074242059335583
median Δmag AP2-AP1   : -0.008030954196522664
NMAD Δmag             : 0.01483839107381566
16-84% Δmag           : [-0.02211263  0.00715697]
median error AP2/AP1  : 0.9981963886251306

Z-band catalogue comparison
AP1 objects           : 13266
AP2 objects           : 13208
matched               : 12843
median separation mas : 9.449883785928073

All valid matched sources
-------------

### 3. Validated PSF-photometry results

The validated one-patch run gives:

| Band | Mutual matches | Median separation (mas) | High-S/N sources | Median AP2/AP1 flux | Median AP2−AP1 (mag) | NMAD (mag) | Median error ratio |
|---|---:|---:|---:|---:|---:|---:|---:|
| Y | 12,842 | 6.916 | 2,683 | 1.00742 | −0.00803 | 0.01484 | 0.99820 |
| Z | 12,843 | 9.450 | 3,102 | 1.03691 | −0.03935 | 0.01687 | 0.99374 |
| J | 12,839 | 0.547 | 1,837 | 1.00003 | −0.00003 | 0.00666 | 0.99851 |

J agrees essentially exactly between the two runs, as expected for a band whose reference photometry is unchanged. This demonstrates that the rerun itself does not introduce a general photometric shift. The small Y offset and the larger Z offset are therefore associated with the changed Z/Y reference calibration in AP2. Formal PSF-flux uncertainties remain nearly unchanged between the two runs.

### 4. Z aperture photometry compared with VIDEO DR5

The Z-band shift is checked with a measurement closer to the public VIDEO photometry. The LSST-VISTA `base_CircularApertureFlux_6_0` measurement is compared with VIDEO DR5 `Z_FLUX_APER_2`, following the aperture choice used in the DP1 photometry diagnostics.

The AP1/AP2 mutual Z matches from the previous section are matched to VIDEO within 0.5 arcsec. The comparison sample requires:

- valid, positive aperture fluxes and uncertainties in AP1, AP2 and VIDEO;
- no LSST-VISTA aperture-flux flag;
- `Z_FLAGS == 0` in VIDEO;
- `Z_CLASS_STAR > 0.8`; and
- S/N ≥ 20 in AP1, AP2 and VIDEO.

VIDEO catalogue fluxes use zeropoint 30.0 and are converted to nJy before comparison. The absolute pipeline−VIDEO residual should not be interpreted as an absolute zero-point measurement because the two catalogues use different image processing and photometric measurement procedures.

In [3]:
z = comparisons["Z"]
cat1_z = z["cat1"]
cat2_z = z["cat2"]
m1 = z["m1"]
m2 = z["m2"]

# LSST-VISTA 6-pixel-radius circular aperture.
f1 = np.asarray(
    cat1_z["base_CircularApertureFlux_6_0_instFlux"],
    dtype=float,
)[m1]
e1 = np.asarray(
    cat1_z["base_CircularApertureFlux_6_0_instFluxErr"],
    dtype=float,
)[m1]
q1 = np.asarray(
    cat1_z["base_CircularApertureFlux_6_0_flag"],
    dtype=bool,
)[m1]

f2 = np.asarray(
    cat2_z["base_CircularApertureFlux_6_0_instFlux"],
    dtype=float,
)[m2]
e2 = np.asarray(
    cat2_z["base_CircularApertureFlux_6_0_instFluxErr"],
    dtype=float,
)[m2]
q2 = np.asarray(
    cat2_z["base_CircularApertureFlux_6_0_flag"],
    dtype=bool,
)[m2]

# Coordinates of the AP2 side of the already-established AP1/AP2 matches.
coord_ap2 = SkyCoord(
    ra=np.asarray(cat2_z["coord_ra"], dtype=float)[m2] * u.rad,
    dec=np.asarray(cat2_z["coord_dec"], dtype=float)[m2] * u.rad,
)

# Load only the VIDEO quantities needed here.
with fits.open(VIDEO_PATH, memmap=True) as hdul:
    data = hdul[1].data

    video_ra = np.asarray(data["RA2000"], dtype=float)
    video_dec = np.asarray(data["DEC2000"], dtype=float)
    video_flux = np.asarray(data["Z_FLUX_APER_2"], dtype=float)
    video_err = np.asarray(data["Z_FLUXERR_APER_2"], dtype=float)
    video_flag = np.asarray(data["Z_FLAGS"])
    video_class = np.asarray(data["Z_CLASS_STAR"], dtype=float)

valid_coord = np.isfinite(video_ra) & np.isfinite(video_dec)
video_rows = np.flatnonzero(valid_coord)

coord_video = SkyCoord(
    ra=video_ra[valid_coord] * u.deg,
    dec=video_dec[valid_coord] * u.deg,
)

idx_video, sep_video, _ = coord_ap2.match_to_catalog_sky(coord_video)
vrow = video_rows[idx_video]

# VIDEO catalogue fluxes have AB zeropoint 30.0; nJy has zeropoint 31.4.
VIDEO_TO_NJY = 10 ** ((31.4 - 30.0) / 2.5)

vf = video_flux[vrow] * VIDEO_TO_NJY
ve = video_err[vrow] * VIDEO_TO_NJY
vflag = video_flag[vrow]
vclass = video_class[vrow]

sn1 = f1 / e1
sn2 = f2 / e2
snv = vf / ve

good = (
    (sep_video < MATCH_RADIUS)
    & np.isfinite(f1)
    & np.isfinite(f2)
    & np.isfinite(e1)
    & np.isfinite(e2)
    & np.isfinite(vf)
    & np.isfinite(ve)
    & (f1 > 0)
    & (f2 > 0)
    & (vf > 0)
    & (e1 > 0)
    & (e2 > 0)
    & (ve > 0)
    & (~q1)
    & (~q2)
    & (vflag == 0)
    & (vclass > 0.8)
    & (sn1 >= SNR_MIN)
    & (sn2 >= SNR_MIN)
    & (snv >= SNR_MIN)
)

dmag_ap1_video = -2.5 * np.log10(f1[good] / vf[good])
dmag_ap2_video = -2.5 * np.log10(f2[good] / vf[good])
dmag_ap2_ap1 = -2.5 * np.log10(f2[good] / f1[good])

print("Z aperture-6 vs VIDEO APER_2")
print("----------------------------")
print("N sources:", good.sum())

for label, values in [
    ("AP1 aperture6 - VIDEO APER2", dmag_ap1_video),
    ("AP2 aperture6 - VIDEO APER2", dmag_ap2_video),
    ("AP2 aperture6 - AP1 aperture6", dmag_ap2_ap1),
]:
    med, nmad, p = robust_summary(values)

    print(f"\n{label}")
    print("-" * len(label))
    print("median Δmag :", med)
    print("NMAD        :", nmad)
    print("16-84%      :", p)

Z aperture-6 vs VIDEO APER_2
----------------------------
N sources: 346

AP1 aperture6 - VIDEO APER2
---------------------------
median Δmag : 0.28591755413655917
NMAD        : 0.04121589829133961
16-84%      : [0.23345829 0.32263341]

AP2 aperture6 - VIDEO APER2
---------------------------
median Δmag : 0.2501853110787969
NMAD        : 0.03674142533232718
16-84%      : [0.19558797 0.28723211]

AP2 aperture6 - AP1 aperture6
-----------------------------
median Δmag : -0.0377740004031753
NMAD        : 0.010651547628982305
16-84%      : [-0.05142191 -0.02303225]


### 5. Validated Z/VIDEO result and conclusion

For the common clean stellar sample of 346 sources:

| Comparison | Median Δmag | NMAD (mag) |
|---|---:|---:|
| AP1 aperture-6 − VIDEO `APER_2` | +0.28592 | 0.04122 |
| AP2 aperture-6 − VIDEO `APER_2` | +0.25019 | 0.03674 |
| AP2 aperture-6 − AP1 aperture-6 | −0.03777 | 0.01065 |

The aperture comparison independently confirms the approximately 0.04 mag Z shift already seen with PSF photometry. AP2 moves the LSST-VISTA Z measurement by about 0.036 mag closer to the VIDEO `APER_2` measurement on the same source sample, while the robust scatter is slightly smaller. The remaining absolute pipeline−VIDEO offset is not interpreted as a zero-point error because the measurements are not identical.

Together with the unchanged J control band, the small Y shift, the stable formal uncertainties and the consistent Z behaviour in both PSF and circular-aperture measurements, these tests support using **Approach 2 for the full DP2 ELAIS-S1 run**.